# ODI to Databricks Migration
**Source:** Dept_pkg / Increment
**Target:** workspace.odi_trg.trg_employees
**Generated:** 2024-05-22

In [ ]:
dbutils.widgets.text("v_lst_dt", "")
dbutils.widgets.text("ODI_SESS_NO", "174d85e3-1541-4d19-aa8b-5cd0304d5247")
dbutils.widgets.text("ETL_PROC_WID", "-1")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1")

### ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get last run date
CREATE OR REPLACE TEMPORARY VIEW v_last_run_dt AS
SELECT COALESCE(
    (SELECT date_format(last_run_dt, 'dd-MM-yy') 
     FROM workspace.odi_trg.log_table1 
     WHERE pkg_name = 'Dept_pkg'),
    date_format(current_timestamp(), 'dd-MM-yy')
) AS last_run_dt;

In [ ]:
%sql
-- SCEN_TASK_NO {2}: Update log table
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

### Staging Table

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop staging
DROP TABLE IF EXISTS workspace.odi_trg.c_0agequ5fm87vqshdc66m9g3qjor;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create staging
CREATE TABLE workspace.odi_trg.c_0agequ5fm87vqshdc66m9g3qjor
(
    EMPLOYEE_ID BIGINT,
    FIRST_NAME STRING,
    LAST_NAME STRING,
    EMAIL STRING,
    PHONE_NUMBER STRING,
    HIRE_DATE TIMESTAMP,
    JOB_ID STRING,
    SALARY DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID BIGINT,
    DEPARTMENT_ID BIGINT,
    LAST_UPD_DT TIMESTAMP
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Load staging from source
INSERT INTO workspace.odi_trg.c_0agequ5fm87vqshdc66m9g3qjor
SELECT 
    SRC_EMPLOYEES_1.EMPLOYEE_ID,
    SRC_EMPLOYEES_1.FIRST_NAME,
    SRC_EMPLOYEES_1.LAST_NAME,
    SRC_EMPLOYEES_1.EMAIL,
    SRC_EMPLOYEES_1.PHONE_NUMBER,
    SRC_EMPLOYEES_1.HIRE_DATE,
    SRC_EMPLOYEES_1.JOB_ID,
    SRC_EMPLOYEES_1.SALARY,
    SRC_EMPLOYEES_1.COMMISSION_PCT,
    SRC_EMPLOYEES_1.MANAGER_ID,
    SRC_EMPLOYEES_1.DEPARTMENT_ID,
    SRC_EMPLOYEES_1.LAST_UPD_DT
FROM workspace.odi_src.src_employees AS SRC_EMPLOYEES_1
WHERE SRC_EMPLOYEES_1.LAST_UPD_DT >= to_date('${v_lst_dt}', 'dd-MM-yy');

### Flow Table

In [ ]:
%sql
-- SCEN_TASK_NO {70}: Drop flow table
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_employees_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {80}: Create flow table
CREATE TABLE workspace.odi_trg.i_trg_employees_flow
(
    EMPLOYEE_ID BIGINT,
    FIRST_NAME STRING,
    LAST_NAME STRING,
    EMAIL STRING,
    PHONE_NUMBER STRING,
    HIRE_DATE TIMESTAMP,
    JOB_ID STRING,
    SALARY DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID BIGINT,
    DEPARTMENT_ID BIGINT,
    LAST_UPD_DT TIMESTAMP,
    IND_UPDATE STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {90}: Load flow table with detection
INSERT INTO workspace.odi_trg.i_trg_employees_flow
SELECT 
    S.EMPLOYEE_ID,
    S.FIRST_NAME,
    S.LAST_NAME,
    S.EMAIL,
    S.PHONE_NUMBER,
    S.HIRE_DATE,
    S.JOB_ID,
    S.SALARY,
    S.COMMISSION_PCT,
    S.MANAGER_ID,
    S.DEPARTMENT_ID,
    S.LAST_UPD_DT,
    'I' AS IND_UPDATE
FROM workspace.odi_trg.c_0agequ5fm87vqshdc66m9g3qjor AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_employees AS T
    WHERE T.EMPLOYEE_ID = S.EMPLOYEE_ID 
      AND (T.FIRST_NAME <=> S.FIRST_NAME)
      AND (T.LAST_NAME <=> S.LAST_NAME)
      AND (T.EMAIL <=> S.EMAIL)
      AND (T.PHONE_NUMBER <=> S.PHONE_NUMBER)
      AND (T.HIRE_DATE <=> S.HIRE_DATE)
      AND (T.JOB_ID <=> S.JOB_ID)
      AND (T.SALARY <=> S.SALARY)
      AND (T.COMMISSION_PCT <=> S.COMMISSION_PCT)
      AND (T.MANAGER_ID <=> S.MANAGER_ID)
      AND (T.DEPARTMENT_ID <=> S.DEPARTMENT_ID)
      AND (T.LAST_UPD_DT <=> S.LAST_UPD_DT)
);

In [ ]:
%sql
-- SCEN_TASK_NO {100/110}: Optimize Flow
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_employees_flow ZORDER BY (EMPLOYEE_ID);

### Error / Check Tables

In [ ]:
%sql
-- SCEN_TASK_NO {120}: Create SNP_CHECK_TAB
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME STRING,
    SCHEMA_NAME STRING,
    RESOURCE_NAME STRING,
    FULL_RES_NAME STRING,
    ERR_TYPE STRING,
    ERR_MESS STRING,
    CHECK_DATE TIMESTAMP,
    ORIGIN STRING,
    CONS_NAME STRING,
    CONS_TYPE STRING,
    ERR_COUNT BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {130}: Create E$ table
CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_employees
(
    ODI_ROW_ID STRING,
    ODI_ERR_TYPE STRING,
    ODI_ERR_MESS STRING,
    ODI_CHECK_DATE TIMESTAMP,
    EMPLOYEE_ID BIGINT,
    FIRST_NAME STRING,
    LAST_NAME STRING,
    EMAIL STRING,
    PHONE_NUMBER STRING,
    HIRE_DATE TIMESTAMP,
    JOB_ID STRING,
    SALARY DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID BIGINT,
    DEPARTMENT_ID BIGINT,
    LAST_UPD_DT TIMESTAMP,
    ODI_ORIGIN STRING,
    ODI_CONS_NAME STRING,
    ODI_CONS_TYPE STRING,
    ODI_PK STRING,
    ODI_SESS_NO STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {140}: Cleanup previous errors
DELETE FROM workspace.odi_trg.e_trg_employees
WHERE (ODI_ERR_TYPE = 'S' AND 1 = 0)
   OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(4)test.Increment');

### Constraints Violation Detection

In [ ]:
%sql
-- SCEN_TASK_NO {150/160/170/180}: Log Not Null Violations
INSERT INTO workspace.odi_trg.e_trg_employees
(
    ODI_PK, ODI_SESS_NO, ODI_ROW_ID, ODI_ERR_TYPE, ODI_ERR_MESS, ODI_CHECK_DATE, 
    ODI_ORIGIN, ODI_CONS_NAME, ODI_CONS_TYPE, EMPLOYEE_ID, FIRST_NAME, LAST_NAME, 
    EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, 
    DEPARTMENT_ID, LAST_UPD_DT
)
SELECT
    uuid(), '${ODI_SESS_NO}', CAST(monotonically_increasing_id() AS STRING), 'F', 
    CONCAT('ODI-15066: The column ', err_col, ' cannot be null.'), 
    current_timestamp(), '(4)test.Increment', err_col, 'NN', 
    EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, 
    SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
FROM (
    SELECT *, CASE 
        WHEN LAST_NAME IS NULL THEN 'LAST_NAME'
        WHEN EMAIL IS NULL THEN 'EMAIL'
        WHEN HIRE_DATE IS NULL THEN 'HIRE_DATE'
        WHEN JOB_ID IS NULL THEN 'JOB_ID'
        ELSE NULL END AS err_col
    FROM workspace.odi_trg.i_trg_employees_flow
) WHERE err_col IS NOT NULL;

In [ ]:
%sql
-- SCEN_TASK_NO {200}: Remove invalid records from flow using PK join
MERGE INTO workspace.odi_trg.i_trg_employees_flow AS T
USING workspace.odi_trg.e_trg_employees AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
AND S.ODI_SESS_NO = '${ODI_SESS_NO}'
WHEN MATCHED THEN DELETE;

In [ ]:
%sql
-- SCEN_TASK_NO {210}: Insert summary into snp_check_tab
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME, RESOURCE_NAME, FULL_RES_NAME, ERR_TYPE, ERR_MESS, 
    CHECK_DATE, ORIGIN, CONS_NAME, CONS_TYPE, ERR_COUNT
)
SELECT 
    'ODI_TRG', 'TRG_EMPLOYEES', 'workspace.odi_trg.trg_employees', 
    E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, 
    E.ODI_CONS_NAME, E.ODI_CONS_TYPE, COUNT(1)
FROM workspace.odi_trg.e_trg_employees AS E
WHERE E.ODI_ERR_TYPE = 'F'
AND E.ODI_ORIGIN = '(4)test.Increment'
GROUP BY E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, E.ODI_CONS_NAME, E.ODI_CONS_TYPE;

### Mark Records for Update

In [ ]:
%sql
-- SCEN_TASK_NO {220}: Identify existing records for update
MERGE INTO workspace.odi_trg.i_trg_employees_flow AS T
USING (
    SELECT EMPLOYEE_ID
    FROM workspace.odi_trg.trg_employees
) AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

### MERGE into Target

In [ ]:
%sql
-- SCEN_TASK_NO {260/270}: Combined Merge Target
MERGE INTO workspace.odi_trg.trg_employees AS T
USING workspace.odi_trg.i_trg_employees_flow AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.FIRST_NAME     = S.FIRST_NAME,
    T.LAST_NAME      = S.LAST_NAME,
    T.EMAIL          = S.EMAIL,
    T.PHONE_NUMBER   = S.PHONE_NUMBER,
    T.HIRE_DATE      = S.HIRE_DATE,
    T.JOB_ID         = S.JOB_ID,
    T.SALARY         = S.SALARY,
    T.COMMISSION_PCT = S.COMMISSION_PCT,
    T.MANAGER_ID     = S.MANAGER_ID,
    T.DEPARTMENT_ID  = S.DEPARTMENT_ID,
    T.LAST_UPD_DT    = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT
(
    EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, 
    HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, 
    DEPARTMENT_ID, LAST_UPD_DT
) VALUES (
    S.EMPLOYEE_ID, S.FIRST_NAME, S.LAST_NAME, S.EMAIL, S.PHONE_NUMBER, 
    S.HIRE_DATE, S.JOB_ID, S.SALARY, S.COMMISSION_PCT, S.MANAGER_ID, 
    S.DEPARTMENT_ID, S.LAST_UPD_DT
);

### Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {330/350}: Drop temp tables
DROP TABLE IF EXISTS workspace.odi_trg.c_0agequ5fm87vqshdc66m9g3qjor;
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_employees_flow;